# GradientTape

In **TensorFlow**, `tf.GradientTape` is an **automatic differentiation** tool used to compute gradients, which are essential for training machine learning models. During training, a model learns by minimizing a **loss function**, and gradients indicate how each model parameter (weights and biases) should be adjusted to reduce that loss.

`tf.GradientTape` records operations performed on tensors during the **forward pass** and then automatically computes the gradients during the **backward pass** using the **chain rule** from calculus.

Automatic differentiation is one of TensorFlow's core features and enables efficient implementation of algorithms such as **Gradient Descent**, **Stochastic Gradient Descent (SGD)**, and **Adam**.

---

# Why are Gradients Important?

A neural network learns by repeatedly performing the following steps:

1. Make a prediction (**forward pass**).
2. Compare the prediction with the true value by computing a **loss**.
3. Calculate how much each parameter contributed to the error (**backward pass**).
4. Update the parameters to reduce the loss.
5. Repeat until the model converges.

Without gradients, TensorFlow would have no information about **how to improve the model's parameters**.

---

# How `tf.GradientTape` Works

## 1. Recording Operations

When code is executed inside a

```python
with tf.GradientTape() as tape:
```

block, TensorFlow records every mathematical operation involving tensors that are:

- Created using `tf.Variable()`.
- Explicitly watched using `tape.watch()`.

Variables are automatically tracked because they represent the trainable parameters of a machine learning model.

Example:

```python
x = tf.Variable(3.0)

with tf.GradientTape() as tape:
    y = x ** 2
```

TensorFlow records the operation:

```text
y = x²
```

---

## 2. Computing Gradients

After the forward computation is complete, gradients are calculated using:

```python
gradient = tape.gradient(target, sources)
```

where:

- **target** is usually the loss function.
- **sources** are the variables with respect to which the gradients are computed.

Example:

```python
x = tf.Variable(3.0)

with tf.GradientTape() as tape:
    y = x ** 2

gradient = tape.gradient(y, x)

print(gradient)
```

Output:

```text
tf.Tensor(6.0, shape=(), dtype=float32)
```

Mathematically:

\[
y = x^2
\]

\[
\frac{dy}{dx} = 2x
\]

Since:

\[
x = 3
\]

The gradient is:

\[
2 \times 3 = 6
\]

---

## 3. Applying Gradients

The computed gradients are then used by an **optimizer** to update the model's trainable variables.

Example:

```python
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

optimizer.apply_gradients(
    zip(gradients, model.trainable_variables)
)
```

The optimizer adjusts the model parameters according to the chosen optimization algorithm.

Common TensorFlow optimizers include:

- `tf.keras.optimizers.SGD`
- `tf.keras.optimizers.Adam`
- `tf.keras.optimizers.RMSprop`
- `tf.keras.optimizers.Adagrad`

---

# Basic Example

```python
import tensorflow as tf

# Create a trainable variable
x = tf.Variable(3.0)

# Record operations
with tf.GradientTape() as tape:
    y = x ** 2

# Compute the gradient
gradient = tape.gradient(y, x)

print("x:", x.numpy())
print("y:", y.numpy())
print("Gradient:", gradient.numpy())
```

Output:

```text
x: 3.0
y: 9.0
Gradient: 6.0
```

---

# Computing Multiple Gradients

`GradientTape` can compute gradients for multiple variables simultaneously.

```python
x = tf.Variable(2.0)
y = tf.Variable(4.0)

with tf.GradientTape() as tape:
    z = x**2 + y**3

gradients = tape.gradient(z, [x, y])

print("dz/dx =", gradients[0].numpy())
print("dz/dy =", gradients[1].numpy())
```

Output:

```text
dz/dx = 4.0
dz/dy = 48.0
```

---

# Watching Non-Variable Tensors

Constants are **not** automatically tracked.

If gradients are needed for a constant tensor, it must be explicitly watched.

```python
x = tf.constant(3.0)

with tf.GradientTape() as tape:
    tape.watch(x)
    y = x ** 2

gradient = tape.gradient(y, x)

print(gradient.numpy())
```

Output:

```text
6.0
```

Without `tape.watch(x)`, TensorFlow would return `None` because `tf.constant` objects are not trainable.

---

# GradientTape in Neural Network Training

A typical training step follows this sequence:

```text
Input Data
      │
      ▼
Forward Pass
      │
      ▼
Prediction
      │
      ▼
Loss Function
      │
      ▼
GradientTape
      │
      ▼
Compute Gradients
      │
      ▼
Optimizer Updates Weights
      │
      ▼
Repeat
```

This process is repeated for many iterations (epochs) until the model minimizes the loss.

---

# GradientTape vs. Numerical Differentiation

TensorFlow uses **automatic differentiation**, which is different from numerical differentiation.

| Method | Description | Accuracy | Speed |
|---------|-------------|----------|-------|
| Numerical Differentiation | Approximates gradients using small perturbations | Approximate | Slow |
| Symbolic Differentiation | Manipulates mathematical expressions | Exact | Can become complex |
| **Automatic Differentiation (`GradientTape`)** | Applies the chain rule while recording operations | Exact (up to floating-point precision) | Fast |

Automatic differentiation combines the accuracy of symbolic differentiation with the efficiency needed for deep learning.

---

# Summary

`tf.GradientTape` is TensorFlow's automatic differentiation engine and is fundamental to training machine learning models.

Its workflow consists of three main steps:

1. **Recording operations** performed on trainable variables during the forward pass.
2. **Computing gradients** of the loss function with respect to model parameters using `tape.gradient()`.
3. **Applying gradients** through an optimizer, such as `tf.keras.optimizers.Adam`, to update the model's weights and biases.

Together, `tf.GradientTape` and TensorFlow optimizers implement the backpropagation algorithm, enabling neural networks to learn from data by minimizing the loss function over successive training iterations.

In [ ]:
import tensorflow as tf

# Create variables (trainable parameters)
w = tf.Variable(3.0)
b = tf.Variable(2.0)

# Define a simple function: y = wx + b
def compute_loss(x, y_true):
    y_pred = w * x + b
    return tf.reduce_mean((y_true - y_pred) ** 2)  # Mean squared error

# Input data
x = tf.constant([1.0, 2.0, 3.0])
y_true = tf.constant([2.0, 4.0, 6.0])

# Compute gradients
with tf.GradientTape() as tape:
    loss = compute_loss(x, y_true)

# Get gradients of loss w.r.t. variables
gradients = tape.gradient(loss, [w, b])

# Print gradients
print(f"Gradient w.r.t w: {gradients[0].numpy()}")
print(f"Gradient w.r.t b: {gradients[1].numpy()}")


# Key features

1. Persistent vs. Non-persistent tape:
- By default, tf.GradientTape() is non-persistent, meaning it can only be used once.
- If you need multiple gradient calculations, use tf.GradientTape(persistent=True), but you must manually delete it with tape.delet() to free memory.

In [ ]:
with tf.GradientTape(persistent=True) as tape:
    loss = compute_loss(x, y_true)

grad_w = tape.gradient(loss, w)
grad_b = tape.gradient(loss, b)


2. Higher-Order Gradients(Second Derivatives):
- Tf.GradientTape can compute higher-order derivatives by nesting tapes:

In [ ]:
with tf.GradientTape() as tape1:
    with tf.GradientTape() as tape2:
        loss = compute_loss(x, y_true)
    grad_w = tape2.gradient(loss, w)
second_derivative = tape1.gradient(grad_w, w)


# Why use tf.GradientTape

- Custom training loops: Unlike model.fit(), tf.GradientTape gives you more control over training, allowing implementation of custom loss functions and optimizations. 
- Memory Efficiency: It dinamically computes gradients instead of storing them permanently optimizing performance.
- Flexibility: Works with non-keras models, custom TensorFlow layers, and differentiable operations